<a href="https://colab.research.google.com/github/jhuangjennifer/573ChineseEnglishSummarization/blob/jjnhuang-work/notebooks/models/test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [32]:
from transformers import BartTokenizer, BartModel
import torch

tokenizer = BartTokenizer.from_pretrained('facebook/bart-large')

model = BartModel.from_pretrained(
    'facebook/bart-large',
    force_download=True
)

inputs = tokenizer("Hello, my dog is cute", return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

last_hidden_states = outputs.last_hidden_state
print("model shape:", last_hidden_states.shape)

config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/513 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


model shape: torch.Size([1, 8, 1024])


In [33]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
from datasets import Dataset
from transformers import (
    BartTokenizer,
    BartForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)

print("library load")

library load


In [34]:
pip install transformers

model.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

In [35]:
# dataset check
DATASET_NAME = "XSAMSum"   # you can change this "XMediaSum40k"
MODEL_NAME = "facebook/bart-large"
CACHE_DIR = "./hf_cache"
BASE_DIR = f"sample_data"

In [36]:
# dataset load
import os
from datasets import load_dataset

#BASE_DIR = f"../../data/raw/{DATASET_NAME}"

data_files = {
    "train": os.path.join(BASE_DIR, "train.json"),
    "validation": os.path.join(BASE_DIR, "val.json"),
    "test": os.path.join(BASE_DIR, "test.json"),
}

print(data_files)

dataset_dict = load_dataset("json", data_files=data_files)
dataset_dict

{'train': 'sample_data/train.json', 'validation': 'sample_data/val.json', 'test': 'sample_data/test.json'}


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['dialogue', 'summary', 'summary_de', 'summary_zh'],
        num_rows: 14732
    })
    validation: Dataset({
        features: ['dialogue', 'summary', 'summary_de', 'summary_zh'],
        num_rows: 818
    })
    test: Dataset({
        features: ['dialogue', 'summary', 'summary_de', 'summary_zh'],
        num_rows: 819
    })
})

In [37]:
import pandas as pd

# DataFrame
df_train = dataset_dict['train'].to_pandas()

# result
display(df_train.head())

,dialogue,summary,summary_de,summary_zh
0,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...,amanda hat kekse gebacken und wird jerry morge...,阿曼达烤了饼干，明天会给杰瑞带一些。
1,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...,olivia und olivier stimmen bei dieser wahl für...,奥利维亚和奥利维尔在这次选举中要投票给自由党。
2,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...,kim kann die von tim empfohlene pomodoro-techn...,金可能会尝试蒂姆推荐的番茄工作法来完成更多的工作。
3,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...,"edward denkt, dass er in bella verliebt ist. r...",爱德华认为他爱上了贝拉。瑞秋想让爱德华开门。瑞秋在外面。
4,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com...","sam ist verwirrt, weil er mitbekommen hat, wie...",山姆很困惑，因为他无意中听到瑞克抱怨他这个室友。娜欧米觉得山姆应该和瑞克谈谈。山姆不知道该怎么办。


In [38]:
import pandas as pd

# convert into DataFrame
df_train = dataset_dict['train'].to_pandas()
df_val = dataset_dict['validation'].to_pandas()
df_test = dataset_dict['test'].to_pandas()

# result check
print(f"Train rows: {len(df_train)}")
print(f"Validation rows: {len(df_val)}")
print(f"Test rows: {len(df_test)}")

Train rows: 14732
Validation rows: 818
Test rows: 819


In [39]:
display(df_train.head())
display(df_val.head())
display(df_test.head())

,dialogue,summary,summary_de,summary_zh
0,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...,amanda hat kekse gebacken und wird jerry morge...,阿曼达烤了饼干，明天会给杰瑞带一些。
1,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...,olivia und olivier stimmen bei dieser wahl für...,奥利维亚和奥利维尔在这次选举中要投票给自由党。
2,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...,kim kann die von tim empfohlene pomodoro-techn...,金可能会尝试蒂姆推荐的番茄工作法来完成更多的工作。
3,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...,"edward denkt, dass er in bella verliebt ist. r...",爱德华认为他爱上了贝拉。瑞秋想让爱德华开门。瑞秋在外面。
4,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com...","sam ist verwirrt, weil er mitbekommen hat, wie...",山姆很困惑，因为他无意中听到瑞克抱怨他这个室友。娜欧米觉得山姆应该和瑞克谈谈。山姆不知道该怎么办。


,dialogue,summary,summary_de,summary_zh
0,"A: Hi Tom, are you busy tomorrow’s afternoon?\...",A will go to the animal shelter tomorrow to ge...,"a wird morgen ins tierheim gehen, um einen wel...",a明天要去动物收容所给她儿子买一只小狗。他们上周一已经去了收容所，她儿子选了这只小狗。
1,Emma: I’ve just fallen in love with this adven...,Emma and Rob love the advent calendar. Lauren ...,emma und rob lieben den adventskalender. laure...,艾玛和罗伯都很喜欢这个基督降临历。劳伦在日历内放各种物品，譬如小玩具和圣诞装饰品。她的孩子一...
2,Jackie: Madison is pregnant\r\nJackie: but she...,Madison is pregnant but she doesn't want to ta...,"madison ist schwanger, aber sie möchte darüber...",麦迪逊怀孕了，但她不想提及。帕特丽夏·史蒂文斯结婚了，她认为在这之前自己就怀孕了。
3,Marla: <file_photo>\r\nMarla: look what I foun...,Marla found a pair of boxers under her bed.,marla fand unter ihrem bett ein paar unterhosen.,玛拉在床底下发现了两条四角裤。
4,Robert: Hey give me the address of this music ...,Robert wants Fred to send him the address of t...,"robert möchte, dass fred ihm die adresse des m...",罗伯特想让弗雷德把音像店的地址发给他，因为他需要买吉他弦。


,dialogue,summary,summary_de,summary_zh
0,"Hannah: Hey, do you have Betty's number?\nAman...",Hannah needs Betty's number but Amanda doesn't...,"hannah braucht bettys nummer, aber amanda hat ...",汉娜需要贝蒂的电话号码，但阿曼达没有。她得联系拉里。
1,Eric: MACHINE!\r\nRob: That's so gr8!\r\nEric:...,Eric and Rob are going to watch a stand-up on ...,eric und rob werden sich ein stand-up auf yout...,埃里克和罗伯要在youtube上看一场单口相声。
2,"Lenny: Babe, can you help me with something?\r...",Lenny can't decide which trousers to buy. Bob ...,"lenny kann sich nicht entscheiden, welche hose...",莱尼无法决定买哪条裤子。鲍勃就此给莱尼提了些建议。莱尼听了他的建议，选了质量最好的裤子。
3,"Will: hey babe, what do you want for dinner to...",Emma will be home soon and she will let Will k...,emma kommt bald nach hause sein und sie wird e...,艾玛很快就会回家，而且她会告诉威尔。
4,"Ollie: Hi , are you in Warsaw\r\nJane: yes, ju...",Jane is in Warsaw. Ollie and Jane has a party....,jane ist in warschau. ollie und jane haben ein...,简在华沙，她和奥利有个聚会。她把重要的日子忘了，本来他们周五会共进午餐。但是奥利无意间给简打...


In [40]:
dfs = {split: dataset.to_pandas() for split, dataset in dataset_dict.items()}

display(dfs['train'].head())

,dialogue,summary,summary_de,summary_zh
0,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...,amanda hat kekse gebacken und wird jerry morge...,阿曼达烤了饼干，明天会给杰瑞带一些。
1,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...,olivia und olivier stimmen bei dieser wahl für...,奥利维亚和奥利维尔在这次选举中要投票给自由党。
2,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...,kim kann die von tim empfohlene pomodoro-techn...,金可能会尝试蒂姆推荐的番茄工作法来完成更多的工作。
3,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...,"edward denkt, dass er in bella verliebt ist. r...",爱德华认为他爱上了贝拉。瑞秋想让爱德华开门。瑞秋在外面。
4,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com...","sam ist verwirrt, weil er mitbekommen hat, wie...",山姆很困惑，因为他无意中听到瑞克抱怨他这个室友。娜欧米觉得山姆应该和瑞克谈谈。山姆不知道该怎么办。


In [41]:
# pick random sample
sample = df_train.sample(1)
print(f"dialogue: {sample['dialogue'].values[0]}")
print(f"summary: {sample['summary'].values[0]}")
print(f"summary_de: {sample['summary_de'].values[0]}")
print(f"summary_z: {sample['summary_zh'].values[0]}")

dialogue: Violet: hi! i came across this Austin's article and i thought that you might find it interesting
Violet: <file_other>
Claire: Hi! :) Thanks, but I've already read it. :)
Claire: But thanks for thinking about me :)
summary: Violet sent Claire Austin's article.
summary_de: violet hat claire austins artikel geschickt.
summary_z: 维奥莱特把奥斯汀的文章发给了克莱尔。


In [42]:
import pandas as pd
import re

# 1. Text cleaning function
def clean_text(text):
    if not isinstance(text, str):
        return text
    text = text.replace('\r', '')          # Remove carriage returns
    text = re.sub(r'\s+', ' ', text)       # Replace multiple spaces with a single space
    return text.strip()                    # Remove leading and trailing spaces

# 2. Integrated preprocessing pipeline function
def preprocess_to_clean_df(dataset_split):
    """
    Takes a Hugging Face Dataset as input, extracts only the 'dialogue' and 'summary' columns,
    and returns a DataFrame with missing values and duplicates removed, and text normalized.
    """
    # Convert to Pandas DataFrame
    df = dataset_split.to_pandas()

    # Select only the necessary columns
    df = df[['dialogue', 'summary', 'summary_zh']].copy()

    # Drop missing values
    df = df.dropna(subset=['dialogue', 'summary', 'summary_zh'])

    # Apply text normalization
    df['dialogue'] = df['dialogue'].apply(clean_text)
    df['summary'] = df['summary'].apply(clean_text)
    df['summary_zh'] = df['summary_zh'].apply(clean_text)

    # Remove duplicate data and reset index
    df = df.drop_duplicates().reset_index(drop=True)

    return df

# 3. Create DataFrames with updated variable names
train_data = preprocess_to_clean_df(dataset_dict['train'])
val_data = preprocess_to_clean_df(dataset_dict['validation'])
test_data = preprocess_to_clean_df(dataset_dict['test'])

# 4. Check results
print("Preprocessing results!")
print(f"train_data: {train_data.shape}")
print(f"val_data: {val_data.shape}")
print(f"test_data: {test_data.shape}\n")

# Check a sample
display(train_data.head())

Preprocessing results!
train_data: (14732, 3)
val_data: (818, 3)
test_data: (819, 3)



,dialogue,summary,summary_zh
0,Amanda: I baked cookies. Do you want some? Jer...,Amanda baked cookies and will bring Jerry some...,阿曼达烤了饼干，明天会给杰瑞带一些。
1,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...,奥利维亚和奥利维尔在这次选举中要投票给自由党。
2,"Tim: Hi, what's up? Kim: Bad mood tbh, I was g...",Kim may try the pomodoro technique recommended...,金可能会尝试蒂姆推荐的番茄工作法来完成更多的工作。
3,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...,爱德华认为他爱上了贝拉。瑞秋想让爱德华开门。瑞秋在外面。
4,Sam: hey overheard rick say something Sam: i d...,"Sam is confused, because he overheard Rick com...",山姆很困惑，因为他无意中听到瑞克抱怨他这个室友。娜欧米觉得山姆应该和瑞克谈谈。山姆不知道该怎么办。


In [43]:
train_data['dialogue'].loc[0]

"Amanda: I baked cookies. Do you want some? Jerry: Sure! Amanda: I'll bring you tomorrow :-)"

In [44]:
train_data[['dialogue', 'summary', 'summary_zh']].head()

,dialogue,summary,summary_zh
0,Amanda: I baked cookies. Do you want some? Jer...,Amanda baked cookies and will bring Jerry some...,阿曼达烤了饼干，明天会给杰瑞带一些。
1,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...,奥利维亚和奥利维尔在这次选举中要投票给自由党。
2,"Tim: Hi, what's up? Kim: Bad mood tbh, I was g...",Kim may try the pomodoro technique recommended...,金可能会尝试蒂姆推荐的番茄工作法来完成更多的工作。
3,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...,爱德华认为他爱上了贝拉。瑞秋想让爱德华开门。瑞秋在外面。
4,Sam: hey overheard rick say something Sam: i d...,"Sam is confused, because he overheard Rick com...",山姆很困惑，因为他无意中听到瑞克抱怨他这个室友。娜欧米觉得山姆应该和瑞克谈谈。山姆不知道该怎么办。


In [45]:
test_data[['dialogue', 'summary']].head()

,dialogue,summary
0,"Hannah: Hey, do you have Betty's number? Amand...",Hannah needs Betty's number but Amanda doesn't...
1,Eric: MACHINE! Rob: That's so gr8! Eric: I kno...,Eric and Rob are going to watch a stand-up on ...
2,"Lenny: Babe, can you help me with something? B...",Lenny can't decide which trousers to buy. Bob ...
3,"Will: hey babe, what do you want for dinner to...",Emma will be home soon and she will let Will k...
4,"Ollie: Hi , are you in Warsaw Jane: yes, just ...",Jane is in Warsaw. Ollie and Jane has a party....


In [46]:
val_data[['dialogue', 'summary']].head()

,dialogue,summary
0,"A: Hi Tom, are you busy tomorrow’s afternoon? ...",A will go to the animal shelter tomorrow to ge...
1,Emma: I’ve just fallen in love with this adven...,Emma and Rob love the advent calendar. Lauren ...
2,Jackie: Madison is pregnant Jackie: but she do...,Madison is pregnant but she doesn't want to ta...
3,Marla: <file_photo> Marla: look what I found u...,Marla found a pair of boxers under her bed.
4,Robert: Hey give me the address of this music ...,Robert wants Fred to send him the address of t...


In [47]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
from datasets import Dataset, DatasetDict
from transformers import (
    BartTokenizer,
    BartForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)

# 1. Convert Pandas DataFrames to Hugging Face Dataset format
# Utilizing the previously created train_data, val_data, and test_data.
hg_dataset = DatasetDict({
    "train": Dataset.from_pandas(train_data),
    "validation": Dataset.from_pandas(val_data),
    "test": Dataset.from_pandas(test_data)
})

# 2. Load the Model and Tokenizer
model_name = "facebook/bart-large"
tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)

# 3. Tokenization preprocessing function (applying max_length based on the paper)
def tokenize_function(examples):
    model_inputs = tokenizer(
        examples["dialogue"],
        max_length=1024,
        truncation=True
    )
    labels = tokenizer(
        text_target=examples["summary"],
        max_length=150,
        truncation=True
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# 3. Tokenization preprocessing function (applying max_length based on the paper)
def tokenize_function_translated(examples):
    model_inputs = tokenizer(
        examples["dialogue"],
        max_length=1024,
        truncation=True
    )
    labels = tokenizer(
        text_target=examples["summary_zh"],
        max_length=150,
        truncation=True
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Map tokenization to the entire dataset (batched=True for speed optimization)
print("start tokenizing")
tokenized_datasets = hg_dataset.map(tokenize_function, batched=True)
translated_tokenized_datasets = hg_dataset.map(tokenize_function_translated, batched=True)
print("tokenization complete!")

# 4. Set up Data Collator (Dynamic padding within batches)
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

# 5. M4 Pro customized training setup (Implementing the paper's batch size of 24)
training_args = Seq2SeqTrainingArguments(
    output_dir="./bart-dialogue-summary-final",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=2,       # Reduced from 8 to 2
    gradient_accumulation_steps=12,      # Increased to 12 (2 * 12 = 24, same as paper)
    per_device_eval_batch_size=2,        # Reduced for evaluation as well
    gradient_checkpointing=True,         # Crucial: Trades compute for memory
    num_train_epochs=20,
    weight_decay=0.01,
    save_total_limit=3,
    predict_with_generate=True,
    fp16=False,
)

# 6. Initialize Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"], # Add validation set!
    processing_class=tokenizer,
    data_collator=data_collator,
)

print("Ready to do trainer.train()")

Loading weights:   0%|          | 0/513 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


start tokenizing


Map:   0%|          | 0/14732 [00:00<?, ? examples/s]

Map:   0%|          | 0/818 [00:00<?, ? examples/s]

Map:   0%|          | 0/819 [00:00<?, ? examples/s]

Map:   0%|          | 0/14732 [00:00<?, ? examples/s]

Map:   0%|          | 0/818 [00:00<?, ? examples/s]

Map:   0%|          | 0/819 [00:00<?, ? examples/s]

tokenization complete!
Ready to do trainer.train()


In [48]:
# =====================================================================
# STEP 1: SELECT SMALL SUBSET FOR TESTING
# =====================================================================
# Take only the first 500 samples for a quick sanity check
small_train_dataset = tokenized_datasets["train"].select(range(500))
small_eval_dataset = tokenized_datasets["validation"].select(range(100))

# =====================================================================
# STEP 2: MODIFIED TRAINING ARGUMENTS FOR SPEED
# =====================================================================
training_args = Seq2SeqTrainingArguments(
    output_dir="./bart-test-run",
    eval_strategy="no",
    learning_rate=2e-5,

    # Increase batch size slightly to speed up M4 Pro (since we have 48GB)
    per_device_train_batch_size=4,
    gradient_accumulation_steps=6,       # 4 * 6 = 24 (Still effective batch size 24)

    # Use max_steps for a quick exit
    max_steps=50,                        # Stop training exactly at 50 steps
    num_train_epochs=1,                  # Ensure it doesn't run more than 1 epoch

    optim="adafactor",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    logging_steps=10,                    # Log every 10 steps to see progress
    save_strategy="no",                  # Don't waste time saving checkpoints during test
    predict_with_generate=True,
    fp16=False,
    bf16=False,
    report_to="none"
)

# =====================================================================
# STEP 3: INITIALIZE & RUN
# =====================================================================
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,   # Use the sliced small dataset
    processing_class=tokenizer,
    data_collator=data_collator,
)

print(" Starting a high-speed test run (50 steps)...")
trainer.train()

 Starting a high-speed test run (50 steps)...


Step,Training Loss
10,13.127539
20,10.856592
30,9.515625
40,9.687695
50,8.738184


TrainOutput(global_step=50, training_loss=10.385126953125, metrics={'train_runtime': 61.5525, 'train_samples_per_second': 19.496, 'train_steps_per_second': 0.812, 'total_flos': 577220162125824.0, 'train_loss': 10.385126953125, 'epoch': 2.384})

In [49]:
pip install evaluate

In [50]:
pip install rouge_score

In [51]:
# Load ROUGE metric
import evaluate
import numpy as np

rouge = evaluate.load("rouge")

def compute_metrics(eval_preds):
    print("Inside compute metrics")
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    # Decode predictions and labels into text
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    # Replace -100 in the labels as we can't decode them
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Compute ROUGE scores
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    return {k: round(v * 100, 4) for k, v in result.items()}

# Run evaluation on the test set
print(" Evaluating on the test set...")
results = trainer.evaluate(eval_dataset=translated_tokenized_datasets["test"], metric_key_prefix="test", compute_metrics=compute_metrics)
print(results)

 Evaluating on the test set...


{'test_loss': 3.751953125, 'test_runtime': 3.7515, 'test_samples_per_second': 218.315, 'test_steps_per_second': 27.456, 'epoch': 2.384}


In [ ]:
# Initialize translation tokenizer and model: https://huggingface.co/Helsinki-NLP/opus-mt-en-zhhttps://huggingface.co/Helsinki-NLP/opus-mt-en-zh

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

translate_tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-zh")
translate_model = AutoModelForSeq2SeqLM.from_pretrained("Helsinki-NLP/opus-mt-en-zh")

In [57]:
def summarize_dialogue(text):
    # 1. Prepare input - this returns a dict containing 'input_ids' and 'attention_mask'
    inputs = tokenizer(text, return_tensors="pt", max_length=512, truncation=True).to(model.device)

    # 2. Generate summary - explicitly pass both input_ids and attention_mask
    summary_ids = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"], # Crucial for MPS compatibility
        max_length=150,
        num_beams=4,
        early_stopping=True
    )

    # 3. Decode summary
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

    # 4. Translate summary
    tokenized_summary = translate_tokenizer(summary, return_tensors="pt").to(translate_model.device)
    output = translate_model.generate(**tokenized_summary)

    # 5. Decode and return translated summary
    decoded_output = translate_tokenizer.decode(output[0], skip_special_tokens=True)
    return decoded_output

# Test with the sample dialogue again
sample_text = "Yun: Hey, did you finish the training? Yunmin: Yes, it's done! Yun: Great, what's next?"
print(f"Summary: {summarize_dialogue(sample_text)}")

Summary: 云民的训练已经完成了
